# Keypoint theo phần thân

Raw `(T, 76, 3)` từ `extract-keypoint-vslfullfront-raw`. Chưa normalize.

- **0–32** pose MediaPipe, **33** neck = trung điểm hai vai
- **34–75** hai tay, xen kẽ: `wrist_0, wrist_1, indexTip_0, indexTip_1, ...`
- `_0` tay trái người ký, `_1` tay phải. Camera trước mặt thì tay phải nằm bên trái ảnh

Cell dưới đóng gói **một clip đầu** của mỗi gloss và **mọi frame**. Kéo slider hoặc bấm Play để xem chuyển động. Prev/Next đổi gloss. Mũi tên trái/phải từng frame.


In [1]:
import logging
from multiprocessing import get_context
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
from matplotlib import font_manager
from matplotlib.lines import Line2D

logger = logging.getLogger("keypoint_viz")
if not logger.handlers:
    logging.basicConfig(level=logging.INFO, format="%(levelname)s %(message)s")

plt.rcParams.update({
    "figure.facecolor": "white",
    "axes.facecolor": "white",
    "axes.unicode_minus": False,
    "font.family": "Arial Unicode MS",
})

TRAIN_DIR = Path("vsl400-keypoint/train")
if not TRAIN_DIR.is_dir():
    TRAIN_DIR = Path("/Users/hugonguyen/PythonProjects/signlanguage/vsl400-keypoint/train")

# Same order as extract-keypoint-vslfullfront-raw: 33 pose points, then neck.
BODY_LANDMARKS = [
    "nose", "leftEyeInner", "leftEye", "leftEyeOuter", "rightEyeInner", "rightEye", "rightEyeOuter",
    "leftEar", "rightEar", "mouthLeft", "mouthRight", "leftShoulder", "rightShoulder",
    "leftElbow", "rightElbow", "leftWrist", "rightWrist", "leftPinky", "rightPinky",
    "leftIndex", "rightIndex", "leftThumb", "rightThumb",
    "leftHip", "rightHip", "leftKnee", "rightKnee", "leftAnkle", "rightAnkle",
    "leftHeel", "rightHeel", "leftFootIndex", "rightFootIndex", "neck",
]
HAND_LANDMARKS = [
    "wrist", "indexTip", "indexDIP", "indexPIP", "indexMCP",
    "middleTip", "middleDIP", "middlePIP", "middleMCP",
    "ringTip", "ringDIP", "ringPIP", "ringMCP",
    "littleTip", "littleDIP", "littlePIP", "littleMCP",
    "thumbTip", "thumbIP", "thumbMP", "thumbCMC",
]
# Name-major interleave. Suffix _0 is the signer's left hand, _1 the right.
LANDMARKS = BODY_LANDMARKS + [name + suffix for name in HAND_LANDMARKS for suffix in ("_0", "_1")]
if len(LANDMARKS) != 76:
    raise RuntimeError(f"expected 76 landmarks, got {len(LANDMARKS)}")

NAME_TO_INDEX = {name: index for index, name in enumerate(LANDMARKS)}
TITLE_FONT = font_manager.FontProperties(family="Arial Unicode MS", size=8)

# Pose finger stubs (pinky, index, thumb) stay on the arm. The 21-point
# hands start at index 34 and are a separate MediaPipe model.
PARTS = (
    ("face", list(range(0, 11)), "#1f4e79"),
    ("torso", [11, 12, 23, 24, 33], "#5c6b73"),
    ("left_arm", [13, 15, 17, 19, 21], "#c47b2b"),
    ("right_arm", [14, 16, 18, 20, 22], "#2f6f4e"),
    ("left_leg", [25, 27, 29, 31], "#8d6e63"),
    ("right_leg", [26, 28, 30, 32], "#6d4c41"),
    ("left_hand", list(range(34, 76, 2)), "#e07a3d"),
    ("right_hand", list(range(35, 76, 2)), "#3d8b6e"),
)
_covered = [index for _name, indices, _color in PARTS for index in indices]
if sorted(_covered) != list(range(76)) or len(_covered) != 76:
    raise RuntimeError("body parts do not partition the 76 landmarks")

PART_COLOR = [""] * 76
for _name, indices, color in PARTS:
    for index in indices:
        PART_COLOR[index] = color

BODY_BONES = (
    ("nose", "leftEyeInner"), ("leftEyeInner", "leftEye"), ("leftEye", "leftEyeOuter"), ("leftEyeOuter", "leftEar"),
    ("nose", "rightEyeInner"), ("rightEyeInner", "rightEye"), ("rightEye", "rightEyeOuter"), ("rightEyeOuter", "rightEar"),
    ("nose", "mouthLeft"), ("nose", "mouthRight"), ("mouthLeft", "mouthRight"),
    ("nose", "neck"), ("neck", "leftShoulder"), ("neck", "rightShoulder"), ("leftShoulder", "rightShoulder"),
    ("leftShoulder", "leftElbow"), ("leftElbow", "leftWrist"),
    ("rightShoulder", "rightElbow"), ("rightElbow", "rightWrist"),
    ("leftWrist", "leftPinky"), ("leftWrist", "leftIndex"), ("leftWrist", "leftThumb"),
    ("rightWrist", "rightPinky"), ("rightWrist", "rightIndex"), ("rightWrist", "rightThumb"),
    ("leftShoulder", "leftHip"), ("rightShoulder", "rightHip"), ("leftHip", "rightHip"),
    ("leftHip", "leftKnee"), ("leftKnee", "leftAnkle"), ("leftAnkle", "leftHeel"), ("leftHeel", "leftFootIndex"),
    ("rightHip", "rightKnee"), ("rightKnee", "rightAnkle"), ("rightAnkle", "rightHeel"), ("rightHeel", "rightFootIndex"),
)
# Local hand index follows HAND_LANDMARKS, tip back to the wrist, then the palm.
FINGER_CHAINS = (
    (1, 2, 3, 4, 0),
    (5, 6, 7, 8, 0),
    (9, 10, 11, 12, 0),
    (13, 14, 15, 16, 0),
    (17, 18, 19, 20, 0),
    (4, 8, 12, 16),
)


def hand_index(local: int, side: int) -> int:
    return 34 + local * 2 + side


EDGES = [(NAME_TO_INDEX[start], NAME_TO_INDEX[end]) for start, end in BODY_BONES]
for side in (0, 1):
    for chain in FINGER_CHAINS:
        for start, end in zip(chain, chain[1:]):
            EDGES.append((hand_index(start, side), hand_index(end, side)))
EDGES.append((NAME_TO_INDEX["leftWrist"], hand_index(0, 0)))
EDGES.append((NAME_TO_INDEX["rightWrist"], hand_index(0, 1)))


def present(points: np.ndarray) -> np.ndarray:
    return np.abs(points).sum(axis=-1) > 1e-6


def limits_of(points: np.ndarray, pad_ratio: float = 0.14):
    alive = present(points)
    if not np.any(alive):
        return (0.0, 1.0), (1.0, 0.0)
    xy = points[alive, :2]
    low = xy.min(axis=0)
    high = xy.max(axis=0)
    span = np.maximum(high - low, 0.04)
    pad = span * pad_ratio
    return (float(low[0] - pad[0]), float(high[0] + pad[0])), (
        float(high[1] + pad[1]),
        float(low[1] - pad[1]),
    )


def _style_axis(ax) -> None:
    ax.set_xticks([])
    ax.set_yticks([])
    for spine in ax.spines.values():
        spine.set_visible(False)


def draw_skeleton(ax, frame, *, point_size=16, line_width=1.1, body_labels=False, limits=None) -> None:
    alive = present(frame)
    for start, end in EDGES:
        if not (alive[start] and alive[end]):
            continue
        ax.plot(
            [frame[start, 0], frame[end, 0]],
            [frame[start, 1], frame[end, 1]],
            color="#c5ccd1",
            linewidth=line_width,
            solid_capstyle="round",
            zorder=1,
        )
    for _name, indices, color in PARTS:
        keep = [index for index in indices if alive[index]]
        if not keep:
            continue
        picked = frame[keep]
        ax.scatter(picked[:, 0], picked[:, 1], s=point_size, c=color, linewidths=0, zorder=2)
    if body_labels:
        for index in range(34):
            if not alive[index]:
                continue
            ax.annotate(
                str(index),
                (frame[index, 0], frame[index, 1]),
                textcoords="offset points",
                xytext=(2, 2),
                fontsize=6,
                color=PART_COLOR[index],
            )
    xlim, ylim = limits_of(frame) if limits is None else limits
    ax.set_xlim(*xlim)
    ax.set_ylim(*ylim)
    ax.set_aspect("equal", adjustable="box")
    _style_axis(ax)


def draw_hand(ax, frame, side: int) -> None:
    local_count = len(HAND_LANDMARKS)
    indices = [hand_index(local, side) for local in range(local_count)]
    subset = frame[indices]
    alive = present(subset)
    color = PART_COLOR[indices[0]]
    title = "left hand _0 (signer left)" if side == 0 else "right hand _1 (signer right)"
    ax.set_title(title, fontsize=10, color=color, pad=8)
    if not np.any(alive):
        ax.text(0.5, 0.5, "missing", ha="center", va="center", transform=ax.transAxes, fontsize=11)
        ax.set_xlim(0, 1)
        ax.set_ylim(1, 0)
        _style_axis(ax)
        return
    for chain in FINGER_CHAINS:
        for start, end in zip(chain, chain[1:]):
            if not (alive[start] and alive[end]):
                continue
            ax.plot(
                [subset[start, 0], subset[end, 0]],
                [subset[start, 1], subset[end, 1]],
                color="#c5ccd1",
                linewidth=1.2,
                zorder=1,
            )
    ax.scatter(subset[alive, 0], subset[alive, 1], s=28, c=color, linewidths=0, zorder=2)
    center = subset[alive, :2].mean(axis=0)
    for local in range(local_count):
        if not alive[local]:
            continue
        point = subset[local, :2]
        direction = point - center
        norm = float(np.linalg.norm(direction))
        if norm < 1e-6:
            offset = (0, 6)
        else:
            direction = direction / norm
            offset = (float(direction[0] * 12), float(-direction[1] * 12))
        ax.annotate(
            f"{indices[local]} {HAND_LANDMARKS[local]}",
            point,
            textcoords="offset points",
            xytext=offset,
            ha="center",
            va="center",
            fontsize=6,
            color=color,
        )
    xlim, ylim = limits_of(subset, pad_ratio=0.35)
    ax.set_xlim(*xlim)
    ax.set_ylim(*ylim)
    ax.set_aspect("equal", adjustable="box")
    _style_axis(ax)


def part_legend(fig, y: float) -> None:
    handles = []
    labels = []
    for name, indices, color in PARTS:
        handles.append(
            Line2D([0], [0], marker="o", color="none", markerfacecolor=color, markeredgewidth=0, markersize=7)
        )
        if name.endswith("_hand"):
            label = f"{name}  {indices[0]}, {indices[1]}, ... {indices[-1]}"
        else:
            label = f"{name}  {', '.join(str(index) for index in indices)}"
        labels.append(label)
    fig.legend(
        handles,
        labels,
        loc="lower center",
        ncol=4,
        frameon=False,
        bbox_to_anchor=(0.5, y),
        fontsize=8,
    )


def _best_hand_frame(array: np.ndarray):
    alive = np.abs(array).sum(axis=2) > 1e-6
    score = alive[:, 34::2].mean(axis=1) + alive[:, 35::2].mean(axis=1)
    index = int(np.argmax(score))
    return index, np.asarray(array[index], dtype=np.float32)


def _load_job(path: str):
    array = np.load(path)
    frame_index, frame = _best_hand_frame(array)
    return path, frame_index, int(array.shape[0]), frame


_CACHE = None


def load_representatives():
    """One frame per gloss: first clip, frame where the most hand points exist."""
    global _CACHE
    if _CACHE is not None:
        return _CACHE
    gloss_dirs = sorted((path for path in TRAIN_DIR.iterdir() if path.is_dir()), key=lambda path: path.name)
    jobs = []
    for gloss_dir in gloss_dirs:
        files = sorted(gloss_dir.glob("*.npy"))
        if files:
            jobs.append(str(files[0]))
    logger.info("Loading one frame from %d glosses with a process pool", len(jobs))
    with get_context("fork").Pool() as pool:
        loaded = pool.map(_load_job, jobs, chunksize=16)
    rows = []
    for path, frame_index, length, frame in loaded:
        sample = Path(path)
        rows.append({
            "gloss": sample.parent.name,
            "path": sample,
            "frame_index": frame_index,
            "length": length,
            "frame": frame,
        })
    rows.sort(key=lambda row: row["gloss"])
    logger.info("Ready %d glosses", len(rows))
    _CACHE = rows
    return rows


def gloss_names():
    return [row["gloss"] for row in load_representatives()]


def _hand_spread(frame: np.ndarray, side: int) -> float:
    indices = [hand_index(local, side) for local in range(len(HAND_LANDMARKS))]
    points = frame[indices]
    alive = present(points)
    if int(np.count_nonzero(alive)) < 18:
        return 0.0
    span = points[alive, :2].max(axis=0) - points[alive, :2].min(axis=0)
    return float(span[0] * span[1])


def show_part_layout(gloss: str = "An ủi") -> None:
    """Full skeleton plus both hands. Picks the frame where the hands are most open."""
    files = sorted((TRAIN_DIR / gloss).glob("*.npy"))
    if not files:
        raise KeyError(f"gloss not in train: {gloss}")
    best = None
    for sample in files:
        array = np.load(sample)
        alive = np.abs(array).sum(axis=2) > 1e-6
        left = alive[:, 34::2].mean(axis=1)
        right = alive[:, 35::2].mean(axis=1)
        candidates = np.flatnonzero((left > 0.85) & (right > 0.85))
        if candidates.size == 0:
            candidates = np.arange(array.shape[0])
        step = max(1, int(candidates.size // 12))
        for index in candidates[::step]:
            frame = np.asarray(array[int(index)], dtype=np.float32)
            score = _hand_spread(frame, 0) + _hand_spread(frame, 1)
            if best is None or score > best[0]:
                best = (score, sample, int(index), int(array.shape[0]), frame)
    _score, sample, frame_index, length, frame = best
    logger.info("Layout frame %s %s t=%d spread=%.4f", gloss, sample.name, frame_index, _score)
    fig = plt.figure(figsize=(13.2, 7.4))
    grid = fig.add_gridspec(1, 3, width_ratios=[1.15, 1.0, 1.0])
    body_axis = fig.add_subplot(grid[0, 0])
    left_axis = fig.add_subplot(grid[0, 1])
    right_axis = fig.add_subplot(grid[0, 2])
    draw_skeleton(body_axis, frame, point_size=22, line_width=1.3, body_labels=True)
    body_axis.set_title("body 0-33, neck is the shoulder midpoint", fontsize=10, pad=8)
    draw_hand(left_axis, frame, 0)
    draw_hand(right_axis, frame, 1)
    part_legend(fig, y=0.0)
    fig.suptitle(
        f"{gloss}   {sample.name}   frame {frame_index}/{length}",
        fontsize=12,
    )
    fig.subplots_adjust(left=0.03, right=0.99, top=0.88, bottom=0.16, wspace=0.28)
    plt.show()


def show_gloss(gloss: str, sample_index: int = 0, n_frames: int = 8) -> None:
    """Evenly spaced frames of one clip. Same xy limits on every panel."""
    files = sorted((TRAIN_DIR / gloss).glob("*.npy"))
    if not files:
        raise KeyError(f"gloss not in train: {gloss}")
    if sample_index < 0 or sample_index >= len(files):
        raise IndexError(f"{gloss} has {len(files)} clips, sample_index={sample_index}")
    path = files[sample_index]
    array = np.load(path)
    length = int(array.shape[0])
    count = min(n_frames, length)
    indexes = np.linspace(0, length - 1, count, dtype=int)
    frames = [np.asarray(array[int(index)], dtype=np.float32) for index in indexes]
    stacked = np.concatenate([frame[present(frame), :2] for frame in frames if np.any(present(frame))], axis=0)
    low = stacked.min(axis=0)
    high = stacked.max(axis=0)
    span = np.maximum(high - low, 0.04)
    pad = span * 0.12
    limits = (
        (float(low[0] - pad[0]), float(high[0] + pad[0])),
        (float(high[1] + pad[1]), float(low[1] - pad[1])),
    )
    fig, axes = plt.subplots(1, count, figsize=(2.15 * count, 4.6))
    axes = np.atleast_1d(axes)
    for axis, frame, index in zip(axes, frames, indexes):
        draw_skeleton(axis, frame, point_size=14, line_width=1.0, limits=limits)
        axis.set_title(f"t={int(index)}", fontsize=9)
    part_legend(fig, y=0.0)
    fig.suptitle(f"{gloss}   {path.name}   {length} frames   clip {sample_index + 1}/{len(files)}", fontsize=12)
    fig.tight_layout(rect=(0, 0.08, 1, 0.92))
    plt.show()


def show_gloss_pages(page: int | None = None, page_size: int = 24, columns: int = 8) -> None:
    """Every train gloss, or one page. page=0 is the first page."""
    rows = load_representatives()
    if page is None:
        starts = list(range(0, len(rows), page_size))
    else:
        starts = [page * page_size]
    for start in starts:
        chunk = rows[start:start + page_size]
        if not chunk:
            logger.error("Page %s is past the last gloss", page)
            return
        row_count = int(np.ceil(len(chunk) / columns))
        fig, axes = plt.subplots(row_count, columns, figsize=(columns * 1.55, row_count * 2.2))
        flat = np.atleast_1d(axes).ravel()
        for axis in flat:
            axis.axis("off")
        for axis, item in zip(flat, chunk):
            axis.axis("on")
            draw_skeleton(axis, item["frame"], point_size=10, line_width=0.85)
            axis.set_title(item["gloss"], fontproperties=TITLE_FONT, pad=2)
        part_legend(fig, y=0.005)
        last = start + len(chunk)
        fig.suptitle(
            f"Train glosses {start + 1}-{last} / {len(rows)}   first clip, frame with the most hand points",
            fontsize=11,
        )
        fig.tight_layout(rect=(0, 0.045, 1, 0.95))
        logger.info("Drew glosses %d-%d", start + 1, last)
        plt.show()


def _load_slide_clip(path: str):
    array = np.load(path)
    alive = np.abs(array).sum(axis=2) > 1e-6
    xy = np.round(array[:, :, :2] * 1000.0).astype("<i2")
    xy[~alive] = np.int16(-32768)
    return path, np.ascontiguousarray(xy)


def build_slide_html(sample_index: int = 0) -> str:
    """One clip per gloss, every frame, packed for the browser slider."""
    import base64
    import json
    import zlib

    gloss_dirs = sorted((path for path in TRAIN_DIR.iterdir() if path.is_dir()), key=lambda path: path.name)
    jobs = []
    for gloss_dir in gloss_dirs:
        files = sorted(gloss_dir.glob("*.npy"))
        if sample_index < len(files):
            jobs.append(str(files[sample_index]))
    logger.info("Packing %d glosses, clip index %d, all frames", len(jobs), sample_index)
    with get_context("fork").Pool() as pool:
        loaded = pool.map(_load_slide_clip, jobs, chunksize=16)
    loaded.sort(key=lambda item: Path(item[0]).parent.name)

    meta = []
    chunks = []
    offset = 0
    for path, xy in loaded:
        sample = Path(path)
        meta.append({
            "name": sample.parent.name,
            "file": sample.name,
            "frames": int(xy.shape[0]),
            "offset": offset,
        })
        offset += int(xy.shape[0])
        chunks.append(xy.reshape(-1))
    blob = np.concatenate(chunks).astype("<i2").tobytes()
    payload = base64.b64encode(zlib.compress(blob, 9)).decode("ascii")
    part_of = [0] * 76
    for part_index, (_name, indices, _color) in enumerate(PARTS):
        for index in indices:
            part_of[index] = part_index
    meta_json = json.dumps(meta, ensure_ascii=True).replace("<", "\\u003c")
    edges_json = json.dumps(EDGES)
    colors_json = json.dumps([color for _name, _indices, color in PARTS])
    names_json = json.dumps([name for name, _indices, _color in PARTS])
    part_of_json = json.dumps(part_of)
    logger.info("Slide payload %d glosses, %d frames, %d bytes", len(meta), offset, len(payload))
    page = _SLIDE_PAGE
    page = page.replace("@@B64@@", payload)
    page = page.replace("@@META@@", meta_json)
    page = page.replace("@@EDGES@@", edges_json)
    page = page.replace("@@COLORS@@", colors_json)
    page = page.replace("@@NAMES@@", names_json)
    page = page.replace("@@PARTOF@@", part_of_json)
    return page


def show_slide(sample_index: int = 0):
    """Browser player: pick a gloss, scrub every frame of its first clip."""
    html = build_slide_html(sample_index)

    class _Slide:
        def _repr_html_(self):
            return html

        def __repr__(self):
            return "keypoint slide"

    return _Slide()


_SLIDE_PAGE = r"""<div id="kp-root" style="font-family:'Arial Unicode MS',Arial,sans-serif;color:#1c1c1c;max-width:760px;">
  <div style="display:flex;gap:8px;align-items:center;margin-bottom:8px;">
    <input id="kp-filter" type="text" placeholder="lọc gloss" style="width:140px;padding:6px 8px;border:1px solid #d0d0d0;font:inherit;"/>
    <select id="kp-gloss" style="flex:1;padding:6px 8px;border:1px solid #d0d0d0;font:inherit;"></select>
    <button id="kp-prev" type="button" style="padding:6px 10px;border:1px solid #d0d0d0;background:#fff;font:inherit;">Prev</button>
    <button id="kp-next" type="button" style="padding:6px 10px;border:1px solid #d0d0d0;background:#fff;font:inherit;">Next</button>
  </div>
  <div style="display:flex;gap:8px;align-items:center;margin-bottom:6px;">
    <button id="kp-play" type="button" style="padding:6px 12px;border:1px solid #d0d0d0;background:#fff;font:inherit;min-width:72px;">Play</button>
    <input id="kp-frame" type="range" min="0" max="1" value="0" style="flex:1;"/>
    <span id="kp-count" style="min-width:88px;text-align:right;font-size:13px;"></span>
  </div>
  <div id="kp-status" style="font-size:13px;margin-bottom:8px;color:#444;"></div>
  <canvas id="kp-canvas" width="560" height="700" style="width:560px;height:700px;border:1px solid #e4e4e4;display:block;"></canvas>
  <div id="kp-legend" style="display:flex;flex-wrap:wrap;gap:8px 16px;margin-top:10px;font-size:12px;"></div>
</div>
<script>
(async function () {
  const root = document.getElementById("kp-root");
  const glossSelect = document.getElementById("kp-gloss");
  const filterInput = document.getElementById("kp-filter");
  const frameSlider = document.getElementById("kp-frame");
  const playButton = document.getElementById("kp-play");
  const countLabel = document.getElementById("kp-count");
  const statusLabel = document.getElementById("kp-status");
  const canvas = document.getElementById("kp-canvas");
  const legend = document.getElementById("kp-legend");
  const ctx = canvas.getContext("2d");
  const META = @@META@@;
  const EDGES = @@EDGES@@;
  const COLORS = @@COLORS@@;
  const NAMES = @@NAMES@@;
  const PART_OF = @@PARTOF@@;
  const B64 = "@@B64@@";
  const JOINTS = 76;

  function fold(value) {
    return value.normalize("NFD").replace(/\p{M}/gu, "").toLowerCase();
  }

  let samples = null;
  let playing = false;
  let lastTick = 0;

  function fillLegend() {
    legend.replaceChildren();
    NAMES.forEach(function (name, index) {
      const item = document.createElement("span");
      const swatch = document.createElement("span");
      swatch.style.cssText = "display:inline-block;width:8px;height:8px;border-radius:50%;margin-right:6px;background:" + COLORS[index];
      item.appendChild(swatch);
      item.appendChild(document.createTextNode(name));
      legend.appendChild(item);
    });
  }

  function fillGlossOptions() {
    const query = fold(filterInput.value.trim());
    const previous = glossSelect.value;
    glossSelect.replaceChildren();
    META.forEach(function (gloss, index) {
      if (query && fold(gloss.name).indexOf(query) === -1) {
        return;
      }
      const option = document.createElement("option");
      option.value = String(index);
      option.textContent = gloss.name;
      glossSelect.appendChild(option);
    });
    if (previous && glossSelect.querySelector('option[value="' + previous + '"]')) {
      glossSelect.value = previous;
    }
  }

  function currentGloss() {
    const index = Number(glossSelect.value || 0);
    return META[index];
  }

  function boundsFor(gloss) {
    let minX = Infinity;
    let minY = Infinity;
    let maxX = -Infinity;
    let maxY = -Infinity;
    for (let frame = 0; frame < gloss.frames; frame++) {
      const base = (gloss.offset + frame) * JOINTS * 2;
      for (let joint = 0; joint < JOINTS; joint++) {
        const x = samples[base + joint * 2];
        const y = samples[base + joint * 2 + 1];
        if (x === -32768 || y === -32768) {
          continue;
        }
        if (x < minX) minX = x;
        if (y < minY) minY = y;
        if (x > maxX) maxX = x;
        if (y > maxY) maxY = y;
      }
    }
    if (!Number.isFinite(minX)) {
      return {minX: 0, minY: 0, maxX: 1000, maxY: 1000};
    }
    const padX = Math.max(40, (maxX - minX) * 0.12);
    const padY = Math.max(40, (maxY - minY) * 0.12);
    return {minX: minX - padX, minY: minY - padY, maxX: maxX + padX, maxY: maxY + padY};
  }

  const boundCache = new Map();

  function draw() {
    const gloss = currentGloss();
    if (!gloss || !samples) {
      return;
    }
    const frame = Math.min(Number(frameSlider.value) || 0, gloss.frames - 1);
    frameSlider.max = String(Math.max(gloss.frames - 1, 0));
    frameSlider.value = String(frame);
    countLabel.textContent = (frame + 1) + " / " + gloss.frames;
    statusLabel.textContent = gloss.name + "   " + gloss.file + "   frame " + frame + " / " + gloss.frames;
    let box = boundCache.get(gloss.offset);
    if (!box) {
      box = boundsFor(gloss);
      boundCache.set(gloss.offset, box);
    }
    const width = canvas.width;
    const height = canvas.height;
    const spanX = box.maxX - box.minX || 1;
    const spanY = box.maxY - box.minY || 1;
    const scale = Math.min((width - 48) / spanX, (height - 48) / spanY);
    const originX = (width - spanX * scale) / 2;
    const originY = (height - spanY * scale) / 2;
    const mapX = function (value) { return originX + (value - box.minX) * scale; };
    const mapY = function (value) { return originY + (value - box.minY) * scale; };
    ctx.clearRect(0, 0, width, height);
    ctx.lineWidth = 2;
    ctx.lineCap = "round";
    ctx.strokeStyle = "#c5ccd1";
    const base = (gloss.offset + frame) * JOINTS * 2;
    const xs = new Array(JOINTS);
    const ys = new Array(JOINTS);
    const alive = new Array(JOINTS);
    for (let joint = 0; joint < JOINTS; joint++) {
      const x = samples[base + joint * 2];
      const y = samples[base + joint * 2 + 1];
      alive[joint] = x !== -32768 && y !== -32768;
      xs[joint] = x;
      ys[joint] = y;
    }
    ctx.beginPath();
    for (let edge = 0; edge < EDGES.length; edge++) {
      const start = EDGES[edge][0];
      const end = EDGES[edge][1];
      if (!alive[start] || !alive[end]) {
        continue;
      }
      ctx.moveTo(mapX(xs[start]), mapY(ys[start]));
      ctx.lineTo(mapX(xs[end]), mapY(ys[end]));
    }
    ctx.stroke();
    for (let joint = 0; joint < JOINTS; joint++) {
      if (!alive[joint]) {
        continue;
      }
      ctx.fillStyle = COLORS[PART_OF[joint]];
      ctx.beginPath();
      ctx.arc(mapX(xs[joint]), mapY(ys[joint]), 4, 0, Math.PI * 2);
      ctx.fill();
    }
  }

  function setGloss(index) {
    playing = false;
    playButton.textContent = "Play";
    glossSelect.value = String(index);
    const gloss = META[index];
    frameSlider.max = String(Math.max(gloss.frames - 1, 0));
    frameSlider.value = "0";
    draw();
  }

  function stepGloss(delta) {
    const options = Array.from(glossSelect.options);
    if (!options.length) {
      return;
    }
    let position = options.findIndex(function (option) { return option.value === glossSelect.value; });
    if (position < 0) {
      position = 0;
    }
    position = (position + delta + options.length) % options.length;
    setGloss(Number(options[position].value));
  }

  function tick(now) {
    if (!playing) {
      return;
    }
    if (now - lastTick >= 50) {
      lastTick = now;
      const gloss = currentGloss();
      let frame = Number(frameSlider.value) + 1;
      if (frame >= gloss.frames) {
        frame = 0;
      }
      frameSlider.value = String(frame);
      draw();
    }
    requestAnimationFrame(tick);
  }

  fillLegend();
  fillGlossOptions();
  const preferred = META.findIndex(function (gloss) { return gloss.name === "Ấm"; });
  if (preferred >= 0) {
    glossSelect.value = String(preferred);
  }
  statusLabel.textContent = "loading frames";
  const raw = Uint8Array.from(atob(B64), function (char) { return char.charCodeAt(0); });
  const stream = new Blob([raw]).stream().pipeThrough(new DecompressionStream("deflate"));
  const buffer = await new Response(stream).arrayBuffer();
  samples = new Int16Array(buffer);
  glossSelect.addEventListener("change", function () { setGloss(Number(glossSelect.value)); });
  filterInput.addEventListener("input", function () {
    fillGlossOptions();
    draw();
  });
  frameSlider.addEventListener("input", function () {
    playing = false;
    playButton.textContent = "Play";
    draw();
  });
  playButton.addEventListener("click", function () {
    playing = !playing;
    playButton.textContent = playing ? "Pause" : "Play";
    if (playing) {
      lastTick = 0;
      requestAnimationFrame(tick);
    }
  });
  document.getElementById("kp-prev").addEventListener("click", function () { stepGloss(-1); });
  document.getElementById("kp-next").addEventListener("click", function () { stepGloss(1); });
  draw();
})();
</script>
"""


ModuleNotFoundError: No module named 'matplotlib'

In [2]:
# show_slide()
